## Enterprise Knowledge Assistant using RAG

### Objective

To design an AI-powered enterprise knowledge retrieval assistant using Retrieval-Augmented Generation (RAG) concepts for intelligent contextual retrieval from internal engineering documents, SOPs, release workflows, and operational knowledge bases.

### Problem Statement

Enterprise teams often manage large volumes of internal documentation such as SOPs, release procedures, incident workflows, and operational guides. Traditional keyword-based search systems struggle to provide contextual and semantically relevant information quickly.

This project aims to build an AI-powered semantic retrieval assistant capable of:

* understanding semantic meaning
* retrieving contextually relevant information
* improving engineering productivity
* reducing dependency on manual document search

using modern Generative AI and RAG concepts.

### Business Use Case

Organizations require efficient access to internal operational knowledge for:

* release validation
* incident management
* engineering workflows
* onboarding support
* operational troubleshooting

An AI-powered semantic retrieval assistant can improve productivity, accelerate issue resolution, and enhance knowledge accessibility across enterprise teams.

### Technologies Used
* Python
* LangChain
* FAISS
* HuggingFace Embeddings
* Sentence Transformers
* Google Colab

### Key AI Concepts Used

- Retrieval-Augmented Generation (RAG)
- Embeddings
- Semantic Similarity
- Vector Search
- Chunking
- Contextual Retrieval
- Prompt Engineering

### Project Workflow
* Load enterprise documents
* Split documents into smaller chunks
* Generate embeddings for chunks
* Store embeddings in vector database (FAISS)
* Accept user queries
* Perform semantic similarity search
* Retrieve relevant contextual chunks
* Generate grounded contextual response

### RAG Architecture Overview

User Query → Semantic Retrieval → Relevant Context → AI Response Generation

The system retrieves semantically relevant document chunks using embeddings and vector similarity search before generating contextual responses.

In [1]:
!pip install -q langchain langchain-community sentence-transformers faiss-cpu pypdf transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
#Creating a sample Doc

sample_text = """
Release Validation SOP

1. Validate application deployment.
2. Execute smoke testing.
3. Validate API responses.
4. Perform backend SQL validation.
5. Verify CI/CD deployment logs.
6. Confirm production health checks.
7. Notify stakeholders after validation.

Incident Management SOP

1. Capture incident details.
2. Perform root cause analysis.
3. Validate impacted services.
4. Escalate to engineering teams.
5. Track resolution progress.
6. Validate fix deployment.
7. Close incident after verification.
"""

with open("enterprise_docs.txt", "w") as file:
    file.write(sample_text)

print("Enterprise document created successfully")

Enterprise document created successfully


In [3]:
#Load the doc
with open("enterprise_docs.txt", "r") as file:
    documents = file.read()

print(documents)


Release Validation SOP

1. Validate application deployment.
2. Execute smoke testing.
3. Validate API responses.
4. Perform backend SQL validation.
5. Verify CI/CD deployment logs.
6. Confirm production health checks.
7. Notify stakeholders after validation.

Incident Management SOP

1. Capture incident details.
2. Perform root cause analysis.
3. Validate impacted services.
4. Escalate to engineering teams.
5. Track resolution progress.
6. Validate fix deployment.
7. Close incident after verification.



In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_text(documents)

print(chunks)

['Release Validation SOP', '1. Validate application deployment.\n2. Execute smoke testing.\n3. Validate API responses.\n4. Perform backend SQL validation.\n5. Verify CI/CD deployment logs.\n6. Confirm production health checks.', '6. Confirm production health checks.\n7. Notify stakeholders after validation.', 'Incident Management SOP', '1. Capture incident details.\n2. Perform root cause analysis.\n3. Validate impacted services.\n4. Escalate to engineering teams.\n5. Track resolution progress.\n6. Validate fix deployment.', '6. Validate fix deployment.\n7. Close incident after verification.']


In [5]:
#Creating Embeddings

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_2983/1490914021.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_2983/1490914021.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your sett

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
#Creating a Vector DB

from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_texts(chunks, embedding_model)

print("Vector database created successfully")

Vector database created successfully


In [7]:
#User Query

query = "What are the release validation steps?"

#Semantic Retrieval
retrieved_docs = vector_store.similarity_search(query, k=2)

for doc in retrieved_docs:
    print(doc.page_content)
    print("-------------------")

Release Validation SOP
-------------------
1. Validate application deployment.
2. Execute smoke testing.
3. Validate API responses.
4. Perform backend SQL validation.
5. Verify CI/CD deployment logs.
6. Confirm production health checks.
-------------------


In [8]:
#Contextual Response Generation

context = "\n".join([doc.page_content for doc in retrieved_docs])

final_response = f"""
Based on enterprise SOP documents, the release validation process includes:

{context}
"""

print(final_response)


Based on enterprise SOP documents, the release validation process includes:

Release Validation SOP
1. Validate application deployment.
2. Execute smoke testing.
3. Validate API responses.
4. Perform backend SQL validation.
5. Verify CI/CD deployment logs.
6. Confirm production health checks.



In [9]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("-" * 50)

Chunk 1:
Release Validation SOP
--------------------------------------------------
Chunk 2:
1. Validate application deployment.
2. Execute smoke testing.
3. Validate API responses.
4. Perform backend SQL validation.
5. Verify CI/CD deployment logs.
6. Confirm production health checks.
--------------------------------------------------
Chunk 3:
6. Confirm production health checks.
7. Notify stakeholders after validation.
--------------------------------------------------
Chunk 4:
Incident Management SOP
--------------------------------------------------
Chunk 5:
1. Capture incident details.
2. Perform root cause analysis.
3. Validate impacted services.
4. Escalate to engineering teams.
5. Track resolution progress.
6. Validate fix deployment.
--------------------------------------------------
Chunk 6:
6. Validate fix deployment.
7. Close incident after verification.
--------------------------------------------------


In [10]:
for i, doc in enumerate(retrieved_docs):
    print(f"Retrieved Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Retrieved Document 1:
Release Validation SOP
--------------------------------------------------
Retrieved Document 2:
1. Validate application deployment.
2. Execute smoke testing.
3. Validate API responses.
4. Perform backend SQL validation.
5. Verify CI/CD deployment logs.
6. Confirm production health checks.
--------------------------------------------------


In [11]:
print("Final AI Response:")
print(final_response)

Final AI Response:

Based on enterprise SOP documents, the release validation process includes:

Release Validation SOP
1. Validate application deployment.
2. Execute smoke testing.
3. Validate API responses.
4. Perform backend SQL validation.
5. Verify CI/CD deployment logs.
6. Confirm production health checks.



### Future Enhancements

- PDF document support
- Conversational memory
- Streamlit UI integration
- OpenAI API integration
- Multi-document retrieval
- Response evaluation framework

### Conclusion

This project demonstrates how Retrieval-Augmented Generation (RAG) can improve enterprise knowledge retrieval workflows using semantic search and contextual response generation.

By leveraging embeddings and vector similarity search, the system retrieves contextually relevant information instead of relying solely on keyword matching, improving enterprise productivity and knowledge accessibility.

### Business Recommendations

- Integrate enterprise document repositories for centralized knowledge retrieval
- Expand support for PDF and multi-document ingestion
- Implement enterprise-approved LLM integrations
- Introduce response evaluation and hallucination monitoring
- Enhance with conversational memory and workflow automation